# Specter Mutual: zero to a deployed app

**Time:** 40 minutes: a few terminal commands, three notebooks, then a deploy from the App Builder

**What you build:** an AI claims reviewer for Specter Mutual, a fictional insurer that covers paranormal damage. A RocketRide **agent** reads each claim and recommends approve, deny or escalate. A **policy engine**, plain code with no AI in it, holds that recommendation to the rules. A **claims portal** puts it in front of people, and you deploy it to your RocketRide workspace.

**Done when:** your portal opens on cloud.rocketride.ai and shows Specter Mutual reviewing a claim you filed.

The agent recommends and the engine decides: an agent can be talked into things, plain code can't.

| Part | Where | Time | You leave with |
| --- | --- | --- | --- |
| Start | [The guide](https://rocketride-org.github.io/rocketride-workshops/modules/specter-mutual/): terminal commands | 3 min | A RocketRide project with a new app in it, created from nothing |
| 1 | [Project init](01-project-init.ipynb) | 3 min | A connected client and the app's manifest |
| 2 | [Pipeline build](02-pipeline-build.ipynb) | 15 min | An agent that decides, and a policy engine that holds it to the rules |
| 3 | [UI build](03-ui-build.ipynb) | 12 min | The claims portal, live in the App Builder's Preview |
| Deploy | [The guide](https://rocketride-org.github.io/rocketride-workshops/modules/specter-mutual/): the App Builder's Deploy tab | 7 min | The portal live in your workspace |

> Follow the guide first: it has you make a project folder, run `pip install rocketride nbformat`, `rocketride init` and `rocketride app create`, and save these notebooks into that folder. Then open the parts in order. Each part loads this notebook first with `%run`, so its helpers are shared, and everything a part makes is saved in the project folder for the next one.
> You need Python 3.10+, Node 20+ and pnpm 10+, a RocketRide Cloud account, a developer id for your organization, and `ROCKETRIDE_ANTHROPIC_KEY` in your RocketRide Variables (the portal can save it for you).

## Shared helpers

One RocketRide client, the paths every part uses, and two helpers. The client signs in with the `.env` that the guide's terminal commands wrote into your project folder, so open these notebooks from that folder. This cell doesn't touch the network, so `%run` can load it: each part connects with `await connect()` right after.

In [ ]:
# Shared: the client, paths and helpers for every part. Safe to re-run.
import json
from pathlib import Path
from rocketride import RocketRideClient

WORKSPACE = Path.cwd()                         # your project folder: the one you initialized
APP_ROOT = WORKSPACE / "apps" / "specter-mutual"
PIPELINES = WORKSPACE / "pipelines"

if not (WORKSPACE / ".env").exists():
    raise RuntimeError("No .env here. Open this notebook from your project folder, the one you "
                       "initialized (see the guide's Start the project step).")
if "client" not in globals():
    client = RocketRideClient()  # reads ROCKETRIDE_URI and ROCKETRIDE_APIKEY from .env

async def connect():
    """Connect the shared client once. Every part runs this right after loading this notebook."""
    if not client.is_connected():
        await client.connect()
    print("Connected to RocketRide.")

def replay(notebook):
    """Re-run a part's file-writing cells (%%writefile, or cells headed "# Write"). Skip-ahead cells use it."""
    nb = json.loads(Path(notebook).read_text(encoding="utf-8"))
    for cell in nb["cells"]:
        src = "".join(cell["source"])
        if cell["cell_type"] != "code":
            continue
        if src.startswith("%%writefile "):
            first, _, body = src.partition("\n")
            path = Path(first.split(None, 1)[1].strip())
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(body, encoding="utf-8", newline="\n")
            print("Wrote", path.as_posix())
        elif src.startswith("# Write"):
            exec(src, globals())

print("Shared helpers ready.")

## What's next

- **Docs:** [docs.rocketride.org](https://docs.rocketride.org)
- **Help and community:** [RocketRide Discord](https://discord.gg/PMXrtenMsY)
- **Back to the hub:** [RocketRide workshops](https://github.com/rocketride-org/rocketride-workshops)